In [ ]:
# check climatological wind speed using ERA5...

import xarray as xr
import os
import glob
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

os.chdir('/g/data/jk72/af1544/Tony_normprod_standalone/')

filelistu = sorted(glob.glob('/g/data/rt52/era5/single-levels/monthly-averaged/10u/2021/*.nc'))
filelistv = sorted(glob.glob('/g/data/rt52/era5/single-levels/monthly-averaged/10v/2021/*.nc'))

u10m = xr.open_mfdataset(filelistu, combine='by_coords')
v10m = xr.open_mfdataset(filelistv, combine='by_coords')


In [ ]:
u10msub = u10m.isel(latitude=slice(600,720))#.mean(dim="time")
v10msub = v10m.isel(latitude=slice(600,720))#.mean(dim="time")
u10msub["v10"] = v10msub.v10
wind10m = u10msub
u10msub = 0
v10msub = 0 
u10m = 0
v10m = 0


In [ ]:
print(wind10m)

In [ ]:
# Subset just one time step (e.g. January)
ds = wind10m.isel(time=6) # 6 is july.

# Coarsen to reduce resolution (adjust coarsening factors as needed)
ds = ds.coarsen(latitude=2, longitude=2, boundary='trim').mean().load()

# Extract 2D wind arrays
u = ds.u10.values
v = ds.v10.values

# Generate 2D coordinate grids
lon2d, lat2d = np.meshgrid(ds.longitude.values, ds.latitude.values)

# Confirm shapes match
assert u.shape == v.shape == lon2d.shape == lat2d.shape

# Plot
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.SouthPolarStereo()})
ax.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())

ax.streamplot(
    lon2d, lat2d, u, v,
    transform=ccrs.PlateCarree(),
    density=2,
    linewidth=1,
    arrowsize=1
)

ax.coastlines()
ax.gridlines(draw_labels=False)
ax.set_title("10m Wind Streamlines – July 2021")

plt.show()

In [ ]:
ds = wind10m.isel(time=6).sel(
    latitude=slice(-67, -72),
    longitude=slice(70, 85)
).load()

u = ds.u10.values
v = ds.v10.values
lon2d, lat2d = np.meshgrid(ds['longitude'].values, ds['latitude'].values)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': ccrs.SouthPolarStereo()})
ax.set_extent([70, 88, -72, -67], crs=ccrs.PlateCarree())

ax.streamplot(
    lon2d, lat2d, u, v,
    transform=ccrs.PlateCarree(),
    density=2,
    linewidth=1,
    arrowsize=1
)

ax.coastlines(resolution='10m')
ax.set_title("10m Wind Streamlines – July 2021\n(70–80°E, 65–70°S)")

plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd

# Load wind data
ds = wind10m.isel(time=6).sel(
    latitude=slice(-67, -72),
    longitude=slice(70, 85)
).load()

u = ds.u10.values
v = ds.v10.values
lon = ds.longitude.values
lat = ds.latitude.values
lon2d, lat2d = np.meshgrid(lon, lat)

# Load high-res SCAR ADD coastline
coast = gpd.read_file("./add_coastline/add_coastline_high_res_line_v7_10.shp")

# Plot
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': ccrs.SouthPolarStereo()})
ax.set_extent([70, 85, -72, -67], crs=ccrs.PlateCarree())

# Plot streamlines
ax.streamplot(
    lon2d, lat2d, u, v,
    transform=ccrs.PlateCarree(),
    density=2,
    linewidth=1,
    arrowsize=1
)

# Add coastline
coast.plot(ax=ax, transform=ccrs.PlateCarree(), color='black', linewidth=0.5)

# Tidy
ax.set_title("10m Wind Streamlines – July 2021\nwith High-Res SCAR Coastline")
ax.gridlines(draw_labels=False)
plt.show()
